In [ ]:
# Step-1: Setup
import os
import re
import json
import math
import warnings
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
# Step-2: Load data

# If notebook is inside src/
DATA_DIR = Path("../data").resolve()

print("DATA_DIR =", DATA_DIR)

def read_csv(name):
    path = DATA_DIR / name

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    try:
        df = pd.read_csv(path)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin1")

    print(f"{name}: {df.shape}")
    return df


print("\nAvailable CSV files:")
for f in sorted(DATA_DIR.glob("*.csv")):
    print(" -", f.name)

print("\nLoading datasets...\n")

hr = read_csv("hr_employees.csv")
tmc = read_csv("tmc_bookings.csv")
obt = read_csv("obt_searches.csv")
flights = read_csv("flight_segments.csv")
hotels = read_csv("hotel_stays.csv")
cards = read_csv("card_transactions.csv")
expenses = read_csv("expense_reports.csv")
policy_rules = read_csv("travel_policy_rules.csv")
hotel_caps = read_csv("hotel_rate_caps.csv")
airports = read_csv("airports.csv")

print("\nAll datasets loaded successfully.")

In [ ]:
# Step-3: Inspect schemas
tables = {
    "hr": hr,
    "tmc": tmc,
    "obt": obt,
    "flights": flights,
    "hotels": hotels,
    "cards": cards,
    "expenses": expenses,
    "policy_rules": policy_rules,
    "hotel_caps": hotel_caps,
    "airports": airports,
}

for name, df in tables.items():
    print("\n" + "=" * 80)
    print(name, df.shape)
    print(df.columns.tolist())
    display(df.head(3))

In [ ]:
# Step-4: Normalize helpers
def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def norm_email(x):
    return norm_text(x)

def clean_id(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def parse_date_cols(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

def contains_any(text, keywords):
    text = norm_text(text)
    return any(k.lower() in text for k in keywords)

def safe_amount(x):
    try:
        return float(x)
    except Exception:
        return np.nan

In [ ]:
# Step-5: Parse dates
tmc = parse_date_cols(tmc, ["trip_start_date", "trip_end_date"])
obt = parse_date_cols(obt, ["search_date", "departure_date", "return_date"])
flights = parse_date_cols(flights, ["depart_date", "arrive_date"])
hotels = parse_date_cols(hotels, ["check_in", "check_out"])
cards = parse_date_cols(cards, ["transaction_date", "post_date"])
expenses = parse_date_cols(expenses, ["transaction_date", "submitted_date"])

for df, amount_cols in [
    (tmc, ["total_booked_amount_usd"]),
    (obt, ["lowest_airfare_usd", "selected_airfare_usd"]),
    (flights, ["fare_usd"]),
    (hotels, ["nightly_rate_usd"]),
    (cards, ["amount_usd"]),
    (expenses, ["amount_usd"]),
]:
    for col in amount_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
# Step-6: Fuzzy matching helper
try:
    from rapidfuzz import fuzz
    def fuzzy_score(a, b):
        a, b = norm_text(a), norm_text(b)
        if not a or not b:
            return 0.0
        return fuzz.token_sort_ratio(a, b) / 100
except ImportError:
    from difflib import SequenceMatcher
    def fuzzy_score(a, b):
        a, b = norm_text(a), norm_text(b)
        if not a or not b:
            return 0.0
        return SequenceMatcher(None, a, b).ratio()

print("Fuzzy helper ready.")

In [ ]:
# Step-7: HR identity resolver
hr_work = hr.copy()

for col in ["employee_id", "legal_name", "preferred_name", "work_email"]:
    if col not in hr_work.columns:
        hr_work[col] = ""

hr_work["employee_id_norm"] = hr_work["employee_id"].apply(clean_id)
hr_work["work_email_norm"] = hr_work["work_email"].apply(norm_email)
hr_work["legal_name_norm"] = hr_work["legal_name"].apply(norm_text)
hr_work["preferred_name_norm"] = hr_work["preferred_name"].apply(norm_text)

def resolve_employee(name=None, email=None, employee_id=None):
    name = norm_text(name)
    email = norm_email(email)
    employee_id = clean_id(employee_id)

    best = {
        "employee_id": "UNKNOWN",
        "score": 0.0,
        "rationale": "no strong HR match",
    }

    for _, row in hr_work.iterrows():
        score = 0.0
        reasons = []

        if employee_id and employee_id == clean_id(row["employee_id"]):
            score += 0.45
            reasons.append("employee_id")

        if email and email == norm_email(row["work_email"]):
            score += 0.40
            reasons.append("email")

        legal_score = fuzzy_score(name, row["legal_name"])
        pref_score = fuzzy_score(name, row["preferred_name"])
        max_name_score = max(legal_score, pref_score)

        if max_name_score >= 0.92:
            score += 0.20
            reasons.append("strong_name")
        elif max_name_score >= 0.82:
            score += 0.12
            reasons.append("fuzzy_name")

        if score > best["score"]:
            best = {
                "employee_id": row["employee_id"],
                "score": min(score, 1.0),
                "rationale": "+".join(reasons) if reasons else "weak match",
            }

    if best["score"] < 0.25:
        return {
            "employee_id": "UNKNOWN",
            "score": 0.0,
            "rationale": "no strong HR match",
        }

    return best

In [ ]:
# Step-8: Create TMC trip anchors
tmc_work = tmc.copy()

tmc_work["trip_id"] = [f"T{i:04d}" for i in range(1, len(tmc_work) + 1)]

tmc_identity = []

for _, row in tmc_work.iterrows():
    resolved = resolve_employee(
        name=row.get("traveler_name"),
        email=row.get("traveler_email"),
        employee_id=row.get("traveler_employee_id"),
    )

    tmc_identity.append({
        "booking_id": row.get("booking_id"),
        "record_locator": row.get("record_locator"),
        "trip_id": row.get("trip_id"),
        "employee_id": resolved["employee_id"],
        "identity_confidence": resolved["score"],
        "identity_rationale": resolved["rationale"],
    })

tmc_identity = pd.DataFrame(tmc_identity)
tmc_work = tmc_work.merge(tmc_identity, on=["booking_id", "record_locator", "trip_id"], how="left")

display(tmc_work.head())

In [ ]:
# Step-9: Link records to TMC trips
def date_near(d, start, end, buffer_days=5):
    if pd.isna(d) or pd.isna(start) or pd.isna(end):
        return False
    return (start - pd.Timedelta(days=buffer_days)) <= d <= (end + pd.Timedelta(days=buffer_days))

def overlap_window(start1, end1, start2, end2, buffer_days=3):
    if pd.isna(start1) or pd.isna(end1) or pd.isna(start2) or pd.isna(end2):
        return False
    return not (
        end1 < start2 - pd.Timedelta(days=buffer_days)
        or start1 > end2 + pd.Timedelta(days=buffer_days)
    )

def best_tmc_match(record, source):
    candidates = []

    for _, trip in tmc_work.iterrows():
        score = 0.0
        reasons = []

        rec_locator = clean_id(record.get("record_locator", ""))
        booking_id = clean_id(record.get("booking_id", ""))
        trip_locator = clean_id(trip.get("record_locator", ""))
        trip_booking_id = clean_id(trip.get("booking_id", ""))

        if rec_locator and trip_locator and rec_locator == trip_locator:
            score += 0.45
            reasons.append("record_locator")

        if booking_id and trip_booking_id and booking_id == trip_booking_id:
            score += 0.45
            reasons.append("booking_id")

        if source == "flight_segments":
            d = record.get("depart_date")
            if date_near(d, trip.get("trip_start_date"), trip.get("trip_end_date"), 2):
                score += 0.20
                reasons.append("flight_date")

        elif source == "hotel_stays":
            if overlap_window(
                record.get("check_in"),
                record.get("check_out"),
                trip.get("trip_start_date"),
                trip.get("trip_end_date"),
                2
            ):
                score += 0.20
                reasons.append("hotel_dates")

        elif source == "obt_searches":
            hint = clean_id(record.get("booking_ref_hint", ""))
            if hint and trip_locator and hint == trip_locator:
                score += 0.45
                reasons.append("booking_ref_hint")

            if date_near(record.get("departure_date"), trip.get("trip_start_date"), trip.get("trip_end_date"), 5):
                score += 0.15
                reasons.append("departure_date")

        candidates.append({
            "trip_id": trip["trip_id"],
            "employee_id": trip["employee_id"],
            "score": min(score, 1.0),
            "rationale": "+".join(reasons),
        })

    best = max(candidates, key=lambda x: x["score"])

    if best["score"] < 0.35:
        return {
            "trip_id": "NO_TRIP",
            "employee_id": "UNKNOWN",
            "score": 0.0,
            "rationale": "no reliable TMC trip match",
        }

    return best

In [ ]:
# Step-10: Build linked records for TMC, flights, hotels, OBT
linked = []

def add_link(source, record_id, employee_id, trip_id, confidence, rationale):
    linked.append({
        "source": source,
        "record_id": record_id,
        "employee_id": employee_id if employee_id else "UNKNOWN",
        "trip_id": trip_id if trip_id else "NO_TRIP",
        "confidence": round(float(confidence), 2),
        "rationale": rationale,
    })

# TMC bookings
for _, row in tmc_work.iterrows():
    confidence = max(row.get("identity_confidence", 0), 0.75)
    rationale = f"TMC trip anchor; identity={row.get('identity_rationale', '')}"
    add_link(
        "tmc_bookings",
        row.get("booking_id"),
        row.get("employee_id", "UNKNOWN"),
        row.get("trip_id"),
        confidence,
        rationale
    )

# Flights
for _, row in flights.iterrows():
    match = best_tmc_match(row, "flight_segments")
    status = norm_text(row.get("ticket_status"))
    penalty = 0.15 if status in ["canceled", "held"] else 0
    add_link(
        "flight_segments",
        row.get("flight_segment_id", row.get("segment_id", row.get("record_id", ""))),
        match["employee_id"],
        match["trip_id"],
        max(match["score"] - penalty, 0),
        match["rationale"] + (f"; status={status}" if status else "")
    )

# Hotels
for _, row in hotels.iterrows():
    match = best_tmc_match(row, "hotel_stays")
    add_link(
        "hotel_stays",
        row.get("hotel_stay_id", row.get("stay_id", row.get("record_id", ""))),
        match["employee_id"],
        match["trip_id"],
        match["score"],
        match["rationale"]
    )

# OBT searches
for _, row in obt.iterrows():
    match = best_tmc_match(row, "obt_searches")
    action = norm_text(row.get("action"))
    if action in ["abandoned"]:
        match["score"] = min(match["score"], 0.65)
    add_link(
        "obt_searches",
        row.get("search_id", row.get("session_id", row.get("record_id", ""))),
        match["employee_id"],
        match["trip_id"],
        match["score"],
        match["rationale"] + (f"; action={action}" if action else "")
    )

linked_df = pd.DataFrame(linked)
display(linked_df.head(20))

In [ ]:
# Step-11: Link card transactions
travel_mcc_keywords = [
    "air", "airline", "hotel", "lodging", "travel", "taxi", "uber", "lyft",
    "rail", "train", "car rental", "parking"
]

def is_travel_card(row):
    text = " ".join([
        norm_text(row.get("merchant_category", "")),
        norm_text(row.get("merchant_name", "")),
        norm_text(row.get("card_notes", "")),
    ])
    return contains_any(text, travel_mcc_keywords)

def best_card_trip_match(row):
    resolved = resolve_employee(
        name=row.get("employee_name_on_card"),
        email=row.get("employee_email"),
        employee_id=None,
    )

    if not is_travel_card(row):
        return {
            "employee_id": resolved["employee_id"],
            "trip_id": "NO_TRIP",
            "score": 0.80,
            "rationale": "non-travel merchant/category"
        }

    candidates = []

    for _, trip in tmc_work.iterrows():
        score = 0.0
        reasons = []

        if resolved["employee_id"] != "UNKNOWN" and resolved["employee_id"] == trip["employee_id"]:
            score += 0.35
            reasons.append("employee")

        if date_near(row.get("transaction_date"), trip.get("trip_start_date"), trip.get("trip_end_date"), 7):
            score += 0.25
            reasons.append("transaction_date")

        merchant_city = norm_text(row.get("merchant_city"))
        dest = norm_text(trip.get("destination_airport"))
        if merchant_city and merchant_city in norm_text(str(trip.to_dict())):
            score += 0.15
            reasons.append("city_context")

        notes = norm_text(row.get("card_notes"))
        locator = norm_text(trip.get("record_locator"))
        if locator and locator in notes:
            score += 0.30
            reasons.append("locator_in_notes")

        candidates.append({
            "employee_id": trip["employee_id"],
            "trip_id": trip["trip_id"],
            "score": min(score, 1.0),
            "rationale": "+".join(reasons),
        })

    best = max(candidates, key=lambda x: x["score"])

    if best["score"] < 0.45:
        return {
            "employee_id": resolved["employee_id"],
            "trip_id": "NO_TRIP",
            "score": max(0.30, resolved["score"]),
            "rationale": f"travel-like card, no reliable trip match; identity={resolved['rationale']}"
        }

    return best

for _, row in cards.iterrows():
    match = best_card_trip_match(row)
    add_link(
        "card_transactions",
        row.get("card_txn_id"),
        match["employee_id"],
        match["trip_id"],
        match["score"],
        match["rationale"]
    )

linked_df = pd.DataFrame(linked)
display(linked_df[linked_df["source"] == "card_transactions"].head(20))

In [ ]:
# Step-12: Link expense reports
def extract_possible_locator(text):
    text = norm_text(text)
    known = tmc_work["record_locator"].dropna().astype(str).str.lower().unique().tolist()
    hits = [loc for loc in known if loc and loc in text]
    return hits[0] if hits else ""

def best_expense_trip_match(row):
    combined_text = " ".join([
        norm_text(row.get("receipt_text", "")),
        norm_text(row.get("claimed_booking_ref", "")),
        norm_text(row.get("comments", "")),
    ])

    resolved = resolve_employee(
        name=row.get("submitter_name"),
        email=row.get("submitter_email"),
        employee_id=row.get("employee_id_reported"),
    )

    locator_hit = extract_possible_locator(combined_text)

    candidates = []

    for _, trip in tmc_work.iterrows():
        score = 0.0
        reasons = []

        if resolved["employee_id"] != "UNKNOWN" and resolved["employee_id"] == trip["employee_id"]:
            score += 0.35
            reasons.append("employee")

        if locator_hit and locator_hit == norm_text(trip.get("record_locator")):
            score += 0.45
            reasons.append("locator_in_expense_text")

        if date_near(row.get("transaction_date"), trip.get("trip_start_date"), trip.get("trip_end_date"), 7):
            score += 0.20
            reasons.append("transaction_date")

        city = norm_text(row.get("city"))
        if city and city in norm_text(str(trip.to_dict())):
            score += 0.15
            reasons.append("city_context")

        candidates.append({
            "employee_id": trip["employee_id"],
            "trip_id": trip["trip_id"],
            "score": min(score, 1.0),
            "rationale": "+".join(reasons),
        })

    best = max(candidates, key=lambda x: x["score"])

    if best["score"] < 0.45:
        return {
            "employee_id": resolved["employee_id"],
            "trip_id": "NO_TRIP",
            "score": max(0.30, resolved["score"]),
            "rationale": f"expense identity only or no reliable trip match; identity={resolved['rationale']}"
        }

    return best

for _, row in expenses.iterrows():
    match = best_expense_trip_match(row)
    add_link(
        "expense_reports",
        row.get("expense_line_id"),
        match["employee_id"],
        match["trip_id"],
        match["score"],
        match["rationale"]
    )

linked_df = pd.DataFrame(linked)
display(linked_df.tail(20))

In [ ]:
# Step-13: Basic linkage quality checks
print(linked_df.shape)

display(
    linked_df.groupby(["source", "trip_id"])
    .size()
    .reset_index(name="records")
    .sort_values(["source", "records"], ascending=[True, False])
)

display(
    linked_df.groupby("source")["confidence"]
    .describe()
)

In [ ]:
# Step-14: Review item helper
review_items = []

def add_review(category, severity, related_record_ids, summary, recommended_action, confidence):
    review_items.append({
        "rank": None,
        "category": category,
        "severity": severity,
        "related_record_ids": ";".join(related_record_ids),
        "summary": summary,
        "recommended_action": recommended_action,
        "confidence": round(float(confidence), 2),
    })

def src_id(source, rid):
    return f"{source}:{rid}"

In [ ]:
# Step-15: Review: inactive employee travel
if "employment_status" in hr.columns:
    inactive_ids = set(
        hr.loc[
            hr["employment_status"].astype(str).str.lower().ne("active"),
            "employee_id"
        ].astype(str)
    )

    inactive_links = linked_df[
        linked_df["employee_id"].astype(str).isin(inactive_ids)
        & (linked_df["trip_id"] != "NO_TRIP")
    ]

    for emp_id, g in inactive_links.groupby("employee_id"):
        add_review(
            category="inactive_employee",
            severity="High",
            related_record_ids=[src_id(r.source, r.record_id) for r in g.itertuples()],
            summary=f"Travel-related records are linked to inactive employee {emp_id}.",
            recommended_action="Confirm employment status, travel authorization, and whether charges should be reassigned or investigated.",
            confidence=min(0.95, g["confidence"].max())
        )

len(review_items)

In [ ]:
# Step-16: Review: held or not ticketed trips
status_col = "booking_status"

if status_col in tmc.columns:
    held_rows = tmc[
        tmc[status_col].astype(str).str.lower().str.contains("hold|held|not ticketed", na=False)
    ]

    for _, row in held_rows.iterrows():
        rid = row.get("booking_id")
        linked_row = linked_df[
            (linked_df["source"] == "tmc_bookings") &
            (linked_df["record_id"] == rid)
        ]

        related = [src_id("tmc_bookings", rid)]

        add_review(
            category="missing_ticket_or_held_itinerary",
            severity="Medium",
            related_record_ids=related,
            summary=f"TMC booking {rid} appears held or not ticketed.",
            recommended_action="Confirm whether this became a real trip, was abandoned, or should be excluded from spend/trip reporting.",
            confidence=0.85
        )

len(review_items)

In [ ]:
# Step-17: Review: hotel cap violations
if "hotel_city" in hotels.columns and "nightly_rate_usd" in hotels.columns:
    caps_city_col = "city" if "city" in hotel_caps.columns else hotel_caps.columns[0]
    cap_col_candidates = [c for c in hotel_caps.columns if "cap" in c.lower()]
    cap_col = cap_col_candidates[0] if cap_col_candidates else None

    if cap_col:
        hotel_check = hotels.merge(
            hotel_caps,
            left_on=hotels["hotel_city"].astype(str).str.lower(),
            right_on=hotel_caps[caps_city_col].astype(str).str.lower(),
            how="left",
            suffixes=("", "_cap")
        )

        violations = hotel_check[
            hotel_check["nightly_rate_usd"] > hotel_check[cap_col]
        ]

        for _, row in violations.iterrows():
            rid = row.get("hotel_stay_id", row.get("stay_id", row.get("record_id", "")))
            add_review(
                category="hotel_rate_cap_exceeded",
                severity="Medium",
                related_record_ids=[src_id("hotel_stays", rid)],
                summary=f"Hotel nightly rate ${row['nightly_rate_usd']:.2f} exceeds cap ${row[cap_col]:.2f} for {row.get('hotel_city')}.",
                recommended_action="Review for business justification, approved exception, conference rate, or traveler reimbursement adjustment.",
                confidence=0.90
            )

len(review_items)

In [ ]:
# Step-18: Review: duplicate card and expense claims
card_expense_pairs = []

for _, c in cards.iterrows():
    for _, x in expenses.iterrows():
        same_amount = (
            pd.notna(c.get("amount_usd")) and
            pd.notna(x.get("amount_usd")) and
            abs(float(c["amount_usd"]) - float(x["amount_usd"])) <= 1.00
        )

        close_date = False
        if pd.notna(c.get("transaction_date")) and pd.notna(x.get("transaction_date")):
            close_date = abs((c["transaction_date"] - x["transaction_date"]).days) <= 3

        merchant_sim = fuzzy_score(c.get("merchant_name"), x.get("merchant_name"))

        if same_amount and close_date and merchant_sim >= 0.75:
            card_expense_pairs.append((c, x, merchant_sim))

for c, x, sim in card_expense_pairs:
    add_review(
        category="possible_duplicate_expense",
        severity="High",
        related_record_ids=[
            src_id("card_transactions", c.get("card_txn_id")),
            src_id("expense_reports", x.get("expense_line_id")),
        ],
        summary=f"Card transaction and expense line have similar merchant, date, and amount ${c.get('amount_usd')}.",
        recommended_action="Check whether the expense is a reconciliation of the corporate card charge or a duplicate reimbursement request.",
        confidence=0.85
    )

len(review_items)

In [ ]:
# Step-19: Review: refunds and negative card transactions
negative_cards = cards[cards["amount_usd"] < 0] if "amount_usd" in cards.columns else pd.DataFrame()

for _, row in negative_cards.iterrows():
    emp = resolve_employee(
        name=row.get("employee_name_on_card"),
        email=row.get("employee_email")
    )

    possible_original = cards[
        (cards["amount_usd"] > 0) &
        (cards["employee_email"].astype(str).str.lower() == str(row.get("employee_email")).lower()) &
        (cards["merchant_name"].apply(lambda x: fuzzy_score(x, row.get("merchant_name"))) >= 0.70)
    ]

    confidence = 0.75 if len(possible_original) == 0 else 0.55

    add_review(
        category="refund_or_credit_review",
        severity="Medium",
        related_record_ids=[src_id("card_transactions", row.get("card_txn_id"))],
        summary=f"Negative card transaction ${row.get('amount_usd')} may be refund/credit requiring reconciliation.",
        recommended_action="Match credit to original charge, exchange, cancellation, or expense adjustment.",
        confidence=confidence
    )

len(review_items)

In [ ]:
# Step-20: Review: direct travel spend with no trip
card_links = linked_df[
    (linked_df["source"] == "card_transactions") &
    (linked_df["trip_id"] == "NO_TRIP")
]

for _, link in card_links.iterrows():
    card_row = cards[cards["card_txn_id"] == link["record_id"]]
    if card_row.empty:
        continue

    row = card_row.iloc[0]

    if is_travel_card(row):
        add_review(
            category="possible_direct_booking",
            severity="Medium",
            related_record_ids=[src_id("card_transactions", link["record_id"])],
            summary="Travel-like card transaction was not linked to a managed TMC trip.",
            recommended_action="Review whether this was an out-of-channel/direct booking or should be linked to an existing trip.",
            confidence=0.70
        )

len(review_items)

In [ ]:
# Step-21: Rank review items
severity_rank = {
    "High": 3,
    "Medium": 2,
    "Low": 1,
}

review_df = pd.DataFrame(review_items)

if len(review_df) > 0:
    review_df["_severity_rank"] = review_df["severity"].map(severity_rank).fillna(0)
    review_df = review_df.sort_values(
        ["_severity_rank", "confidence"],
        ascending=[False, False]
    ).drop(columns=["_severity_rank"])
    review_df["rank"] = range(1, len(review_df) + 1)

display(review_df.head(30))

In [ ]:
# Step-22: Export required CSVs
# If running inside ~/src/
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required_link_cols = [
    "source",
    "record_id",
    "employee_id",
    "trip_id",
    "confidence",
    "rationale",
]

required_review_cols = [
    "rank",
    "category",
    "severity",
    "related_record_ids",
    "summary",
    "recommended_action",
    "confidence",
]

# Keep only required columns
linked_df = linked_df[required_link_cols].copy()

review_df = (
    review_df[required_review_cols].copy()
    if len(review_df) > 0
    else pd.DataFrame(columns=required_review_cols)
)

# Output file paths
linked_path = OUTPUT_DIR / "linked_records.csv"
review_path = OUTPUT_DIR / "review_items.csv"

# Save CSVs
linked_df.to_csv(linked_path, index=False)
review_df.to_csv(review_path, index=False)

print(f"Wrote: {linked_path.resolve()}")
print(f"Wrote: {review_path.resolve()}")

display(linked_df.head())
display(review_df.head())

In [ ]:
# Step-23: Final validation
print("linked_records.csv")
print(linked_df.shape)
display(linked_df.head(10))

print("\nreview_items.csv")
print(review_df.shape)
display(review_df.head(10))

assert set(required_link_cols).issubset(linked_df.columns)
assert set(required_review_cols).issubset(review_df.columns)

assert linked_df["confidence"].between(0, 1).all()
if len(review_df) > 0:
    assert review_df["confidence"].between(0, 1).all()

print("Validation passed.")